# TRIBE Studio — Colab GPU backend + **ngrok** (official CLI)

This notebook runs **`backend/`** on a **Colab GPU**, starts **uvicorn** on port **8000**, then starts the **official ngrok agent** (`ngrok http 8000`). Your Mac backend uses the printed **`REMOTE_TRIBE_URL`**.

**Security:** never commit **ngrok** or **Hugging Face** tokens in this notebook. If leaked, **revoke** in [ngrok](https://dashboard.ngrok.com/) / [HF tokens](https://huggingface.co/settings/tokens) and create new ones.

**Do not** set `REMOTE_TRIBE_URL` on Colab (infinite loop with the Mac forwarder).

**HF User access token:** Use a Colab Secret **`HF_TOKEN`**, or paste into **`HF_USER_ACCESS_TOKEN`** in cell 3 (same token from [Settings → Access Tokens](https://huggingface.co/settings/tokens); **Read** is enough). **Never commit** a filled `HF_USER_ACCESS_TOKEN` to git. Cell 3 sets env vars and skips slow `login()` unless `TRIBE_HF_LOGIN=1`.

**ngrok:** Cell 5 blocks until the tunnel is ready, then **exits**; **uvicorn + ngrok keep running** in the background. To avoid a silent hang on `getpass`, use Colab Secret **`NGROK_AUTHTOKEN`** or **`NGROK_AUTHTOKEN_MANUAL`** in cell 5 (never commit a real value).

In [1]:
# @title 0) GPU check
import torch
print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a")

cuda: True device: Tesla T4


### 1) Put this repo on the Colab VM

- **Option A:** set `REPO_URL` to your GitHub fork below.
- **Option B:** upload `eureka-hacks.zip` (must contain `backend/`) to `/content/`, set `USE_ZIP=True`.

In [2]:
# @title 1) Clone or unzip
import os, shutil, subprocess
from pathlib import Path
from urllib.parse import quote

REPO_URL = "https://github.com/Doppy258/eureka-hacks.git"  # <-- your fork (must exist on GitHub)
BRANCH = "main"  # if clone fails, we retry without -b (default branch)
ROOT = Path("/content/eureka-hacks")
# If cwd was inside ROOT, deleting ROOT breaks git ("Unable to read current working directory").
_WORKDIR = Path("/content")
_WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(_WORKDIR)

if ROOT.exists():
    shutil.rmtree(ROOT)

USE_ZIP = False

# Private GitHub repo: set Colab secret GITHUB_TOKEN (classic PAT: repo scope) or paste in URL:
#   https://<TOKEN>@github.com/owner/repo.git  — never commit a token; use secrets only.


def _clone(repo_url: str, dest: Path, branch: str | None, *, cwd: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if branch:
        cmd = ["git", "clone", "--depth", "1", "-b", branch, repo_url, str(dest)]
    else:
        cmd = ["git", "clone", "--depth", "1", repo_url, str(dest)]
    p = subprocess.run(cmd, capture_output=True, text=True, cwd=str(cwd))
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout or f"git clone failed ({p.returncode})")


if USE_ZIP:
    z = Path("/content/eureka-hacks.zip")
    if not z.is_file():
        raise FileNotFoundError("Upload eureka-hacks.zip to /content/ or set USE_ZIP=False and fix REPO_URL")
    subprocess.check_call(["unzip", "-q", str(z), "-d", str(ROOT.parent)])
else:
    if "YOUR_GITHUB_USERNAME" in REPO_URL:
        raise RuntimeError("Edit REPO_URL to your real GitHub fork (or set USE_ZIP=True).")

    tok = None
    try:
        from google.colab import userdata

        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    tok = (tok or os.environ.get("GITHUB_TOKEN") or "").strip()
    url = REPO_URL
    if tok and REPO_URL.startswith("https://") and "@" not in REPO_URL.split("://", 1)[1].split("/")[0]:
        url = REPO_URL.replace("https://", f"https://{quote(tok, safe='')}@", 1)

    os.chdir(_WORKDIR)
    try:
        print("Cloning branch", BRANCH, "…")
        _clone(url, ROOT, BRANCH, cwd=_WORKDIR)
    except RuntimeError as e:
        print("Clone with -b failed:", e)
        os.chdir(_WORKDIR)
        if ROOT.exists():
            shutil.rmtree(ROOT, ignore_errors=True)
        print("Retrying clone **without** -b (uses remote default branch)…")
        _clone(url, ROOT, None, cwd=_WORKDIR)

BACKEND = ROOT / "backend"
assert (BACKEND / "main.py").is_file(), f"Missing backend/main.py under {BACKEND}"
print("OK:", BACKEND)

Cloning branch main …
OK: /content/eureka-hacks/backend


In [3]:
# @title 2) Python deps (may take several minutes)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/tribev2.git"])

0

In [4]:
# @title 3) Hugging Face User Access Token for TRIBE (fast — skips slow Hub `login()` by default)
import os, getpass

# Optional: paste your HF *User access token* (read) here for a private Colab session.
# Same token type as https://huggingface.co/settings/tokens — NEVER commit a real value; clear before git push.
HF_USER_ACCESS_TOKEN = ""  # e.g. "hf_..." — never commit a real token

os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ.pop("TRIBE_DEMO", None)

# Optional: Colab → key icon → Secret `HF_TOKEN` (same token; avoids pasting in code).
_secret = None
try:
    from google.colab import userdata

    _secret = userdata.get("HF_TOKEN")
except Exception:
    pass

token = (
    (HF_USER_ACCESS_TOKEN or "").strip()
    or (os.environ.get("HF_TOKEN") or "").strip()
    or (os.environ.get("HUGGING_FACE_HUB_TOKEN") or "").strip()
    or (_secret or "").strip()
)
if not token:
    token = getpass.getpass("HF User Access Token (hidden, hf.co/settings/tokens): ").strip()
if not token:
    raise RuntimeError(
        "Missing token. Create a User Access Token at https://huggingface.co/settings/tokens — set Colab secret HF_TOKEN or export HF_TOKEN / HUGGING_FACE_HUB_TOKEN."
    )

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

# `login()` validates with the Hub and can take a long time on slow networks — not required for downloads.
if os.environ.get("TRIBE_HF_LOGIN", "").strip().lower() in ("1", "true", "yes"):
    try:
        from huggingface_hub import login
    except ImportError:
        import subprocess, sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import login

    login(token=token, add_to_git_credential=False)
    print("HF: ran huggingface_hub.login() (TRIBE_HF_LOGIN=1)")
else:
    print("HF: env tokens set; skipped login. (Set TRIBE_HF_LOGIN=1 if you really need login().)")

HF: env tokens set; skipped login. (Set TRIBE_HF_LOGIN=1 if you really need login().)


In [5]:
# @title 4) Stop old uvicorn / ngrok (re-run friendly)
import subprocess, time
subprocess.run("pkill -f 'uvicorn main:app' || true", shell=True)
subprocess.run("pkill -f '[n]grok' || true", shell=True)
time.sleep(1)

In [6]:
# @title 4b) Install official ngrok CLI (Colab = Debian/Ubuntu)
import shutil, subprocess

if shutil.which("ngrok"):
    print("ngrok already on PATH:", shutil.which("ngrok"))
else:
    print("Installing ngrok via apt (per https://ngrok.com/download Linux / apt ) …")
    install = r'''
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y curl gnupg
curl -fsSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
echo "deb https://ngrok-agent.s3.amazonaws.com buster main" \
  | tee /etc/apt/sources.list.d/ngrok.list >/dev/null
apt-get update -qq
apt-get install -y ngrok
'''
    subprocess.run(["bash", "-lc", install], check=True)
    print("ngrok:", shutil.which("ngrok"))

Installing ngrok via apt (per https://ngrok.com/download Linux / apt ) …
ngrok: /usr/local/bin/ngrok


In [ ]:
# @title 5) Register authtoken, start uvicorn + `ngrok http 8000`, print Mac export
#
# This cell *blocks* until the tunnel URL is found, then *exits*. Uvicorn + ngrok keep running in the
# background until the Colab runtime stops or you run the "pkill" cell.
#
import json, os, shutil, subprocess, sys, time, getpass
from pathlib import Path
from urllib.request import urlopen

print("[1/6] Checking ngrok on PATH…")
if not shutil.which("ngrok"):
    raise RuntimeError("Run cell 4b: ngrok is not on PATH.")

os.chdir(BACKEND)
os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ["TRIBE_DEVICE"] = os.environ.get("TRIBE_DEVICE", "cuda")

# Avoid invisible getpass hangs: Colab secret NGROK_AUTHTOKEN, env, or paste below (never commit).
NGROK_AUTHTOKEN_MANUAL = ""  # optional paste; clear before git push — never commit real tokens
_ng_secret = None
try:
    from google.colab import userdata

    _ng_secret = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass

ng_token = (
    (NGROK_AUTHTOKEN_MANUAL or "").strip()
    or (os.environ.get("NGROK_AUTHTOKEN") or "").strip()
    or (_ng_secret or "").strip()
)
if not ng_token:
    print("No token in env/secret — you will be prompted (if nothing happens, click the notebook input box).")
    ng_token = getpass.getpass("ngrok authtoken (dashboard → Your Authtoken): ").strip()
if not ng_token:
    raise RuntimeError("Missing ngrok authtoken. Add Colab secret NGROK_AUTHTOKEN or set NGROK_AUTHTOKEN_MANUAL.")

print("[2/6] ngrok config add-authtoken …")
subprocess.run(["ngrok", "config", "add-authtoken", ng_token], check=True)

log_path = Path("/content/uvicorn.log")
ngrok_log = Path("/content/ngrok.log")
print("[3/6] Starting uvicorn (stderr+stdout →", log_path, ")…")
log_f = log_path.open("wb")
uv = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
        "--timeout-keep-alive",
        "7200",
    ],
    cwd=str(BACKEND),
    stdout=log_f,
    stderr=subprocess.STDOUT,
)

print("[4/6] Waiting for http://127.0.0.1:8000/api/health (first import can take 1–3 min)…")
uv_ok = False
for i in range(240):
    try:
        urlopen("http://127.0.0.1:8000/api/health", timeout=2)
        uv_ok = True
        print("  uvicorn OK after", i + 1, "s")
        break
    except Exception:
        if i % 15 == 0 and i:
            print("  still waiting…", i, "s (see", log_path, ")")
        time.sleep(1)

if not uv_ok:
    try:
        log_f.flush()
    finally:
        log_f.close()
    print("Uvicorn never answered /api/health. Log tail:")
    print(log_path.read_text(errors="replace")[-12000:])
    raise SystemExit(1)

def _pick_tunnel_base(data):
    """Return https://… or http://… from ngrok inspect JSON (agent versions differ)."""
    tunnels = data.get("tunnels") or []
    https_u = http_u = None
    for t in tunnels:
        u = (t.get("public_url") or t.get("uri") or "").strip()
        if u.startswith("https://"):
            https_u = u.rstrip("/")
        elif u.startswith("http://"):
            http_u = u.rstrip("/")
    return https_u or http_u


print("[5/6] Stop any stale ngrok (frees :4040 inspect API)…")
subprocess.run("pkill -f '[n]grok' || true", shell=True)
time.sleep(2)

print("[5b/6] Starting ngrok http 8000 (log →", ngrok_log, ")…")
ng_log_f = ngrok_log.open("wb")
ng = subprocess.Popen(
    ["ngrok", "http", "8000"],
    stdout=ng_log_f,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

INSPECT = os.environ.get("NGROK_INSPECT", "http://127.0.0.1:4040/api/tunnels")
public_url = None
print("[6/6] Waiting for tunnel URL from", INSPECT, "…")
for i in range(180):
    if ng.poll() is not None:
        try:
            ng_log_f.flush()
        finally:
            ng_log_f.close()
        print("ngrok exited early (code", ng.returncode, "). Log tail:")
        print(ngrok_log.read_text(errors="replace")[-12000:])
        raise SystemExit(1)
    try:
        data = json.load(urlopen(INSPECT, timeout=3))
        public_url = _pick_tunnel_base(data)
    except Exception as e:
        if i % 15 == 0:
            print("  inspect not ready:", repr(e))
    if public_url:
        if public_url.startswith("http://"):
            print("  WARNING: only http:// tunnel listed; Mac may still work. Prefer https if shown in ngrok UI.")
        print("  tunnel OK after", i + 1, "s →", public_url)
        break
    if i % 10 == 0 and i and ngrok_log.exists():
        print("  still waiting…", i, "s — ngrok.log tail:")
        print(ngrok_log.read_text(errors="replace")[-2000:])
    time.sleep(1)

try:
    log_f.flush()
finally:
    log_f.close()
try:
    ng_log_f.flush()
finally:
    ng_log_f.close()

if not public_url:
    print("No HTTPS tunnel. ngrok log tail:")
    print(ngrok_log.read_text(errors="replace")[-8000:])
    print("uvicorn log tail:")
    print(log_path.read_text(errors="replace")[-8000:])
    raise SystemExit(1)

analyze_url = public_url + "/api/analyze"
print("\n--- Cell finished; uvicorn + ngrok still run in the background until runtime disconnects ---")
print("\n--- On your Mac (terminal that runs local uvicorn) ---")
print(f'export REMOTE_TRIBE_URL="{analyze_url}"')
print("unset TRIBE_DEMO")
print("# then: cd backend && source .venv/bin/activate && python -m uvicorn main:app --reload --port 8000")
print("\nTunnel base:", public_url)
print("\nTip: on this VM you can re-print the URL with:")
print("python /content/eureka-hacks/scripts/ngrok_print_analyze_url.py")

[1/6] Checking ngrok on PATH…
[2/6] ngrok config add-authtoken …
[3/6] Starting uvicorn (stderr+stdout → /content/uvicorn.log )…
[4/6] Waiting for http://127.0.0.1:8000/api/health (first import can take 1–3 min)…
  uvicorn OK after 1 s
[5/6] Stop any stale ngrok (frees :4040 inspect API)…
[5b/6] Starting ngrok http 8000 (log → /content/ngrok.log )…
[6/6] Waiting for tunnel URL from http://127.0.0.1:4040/api/tunnels …
  tunnel OK after 1 s → https://vanilla-schilling-bony.ngrok-free.dev

--- Cell finished; uvicorn + ngrok still run in the background until runtime disconnects ---

--- On your Mac (terminal that runs local uvicorn) ---
export REMOTE_TRIBE_URL="https://vanilla-schilling-bony.ngrok-free.dev/api/analyze"
unset TRIBE_DEMO
# then: cd backend && source .venv/bin/activate && python -m uvicorn main:app --reload --port 8000

Tunnel base: https://vanilla-schilling-bony.ngrok-free.dev

Tip: on this VM you can re-print the URL with:
python /content/eureka-hacks/scripts/ngrok_print_an

### VS Code / Cursor Colab extension

1. Install Google’s **Colab** extension.
2. Open this notebook, kernel → **Colab** → **GPU**.
3. Run cells **in order** (0 → 5). Keep the runtime alive while you use the Mac UI.

If `apt-get install ngrok` fails (permissions), use a Colab runtime where you are root, or install ngrok manually in the VM and re-run from cell 4b.